## ESZA019 — Visão Computacional | Trabalho Final | Sistema detector de quedas

Sistema experimental para detecção de pessoas, estimativa de postura e
classificação de eventos compatíveis com queda, desenvolvido com
PyTorch, Torchvision e OpenCV.

A identificação de pessoas é realizada exclusivamente pelo detector neural
SSDLite320 MobileNetV3, utilizando a classe "person" do conjunto COCO.
Informações de movimento, contornos ou subtração de fundo não são empregadas
para confirmar a presença de uma pessoa.

Após a identificação de uma transição postural candidata, o sistema coleta
parâmetros cinemáticos e geométricos durante uma janela temporal fixa. A
classificação do evento considera o tempo de transição, o deslocamento
vertical, a velocidade vertical e a razão entre largura e altura da caixa
delimitadora. A postura observada após o início da análise não determin 
cancela a classificação.

#### Dependências:
    ``pip install torch torchvision opencv-python numpy``

#### Observação:
    Na primeira execução, o Torchvision poderá baixar automaticamente os
    pesos pré-treinados do modite320 MobileNetV3.

In [ ]:
import csv
import datetime as dt
import os
import time
from collections import deque
from pathlib import Path

# Restringe a execução à CPU e reduz mensagens internas não críticas.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"

import cv2
import numpy as np
import torch
from torchvision.models.detection import (
    SSDLite320_MobileNet_V3_Large_Weights,
    ssdlite320_mobilenet_v3_large,
)


# =====================================================================
# Calibração de câmera (Relatório 4 — tabuleiro de xadrez 6x8, webcam do grupo)
# =====================================================================

# Tabuleiro de xadrez impresso pelo grupo e usado na calibração do Lab 4
# (6 cantos internos por coluna, 8 por linha).
CHECKERBOARD_LAB4 = (6, 8)

# Parâmetros intrínsecos já obtidos na calibração do Relatório 4 (item B),
# para a mesma webcam (resolução 640x480) usada neste trabalho final.
# Reaproveitados diretamente, sem necessidade de recalibrar.
CAMERA_MATRIX = np.array([
    [689.75552932,   0.,         319.45913505],
    [  0.,         690.22161692, 244.24211759],
    [  0.,           0.,           1.        ],
])
DIST_COEFFS = np.array([[
    8.94112260e-02, -1.11274089e+00, 1.08907634e-03,
    -1.38399964e-03, 6.12500249e+00,
]])


def corrigir_distorcao(frame):
    """Aplica a correção de distorção da lente (calibração do Lab 4) ao quadro."""
    return cv2.undistort(frame, CAMERA_MATRIX, DIST_COEFFS)



# =====================================================================
# Parâmetros experimentais e configurações do sistema
# =====================================================================

INDICE_CAMERA = 0
FPS_SAIDA = 30.0

# Detecção neural e associação temporal de pessoas
CONFIANCA_MINIMA_PESSOA = 0.65
INTERVALO_DETECCAO = 2
IOU_MINIMO_RASTREAMENTO = 0.10
DISTANCIA_MAXIMA_RASTREAMENTO = 180.0
TEMPO_MAXIMO_SEM_PESSOA = 1.5

# Rejeita detecções incompatíveis com as dimensões esperadas.
ALTURA_MINIMA_PESSOA_FRAC = 0.15
LARGURA_MINIMA_PESSOA_FRAC = 0.05
AREA_MAXIMA_PESSOA_FRAC = 0.80

# Estimativa geométrica de postura e classificação de eventos
CONFIRMA_QUADROS = 5

# Limiares da razão largura/altura da caixa delimitadora.
ASPECTO_EM_PE_MAX = 0.72
ASPECTO_DEITADO_MIN = 1.30

# Utiliza a altura de referência em pé para refinar a classificação.
ALTURA_AGACHADO_MAX_FRAC_EM_PE = 0.78
ALTURA_DEITADO_MAX_FRAC_EM_PE = 0.55

# Limiares utilizados na classificação de eventos de queda.
LIMIAR_TEMPO_QUEDA = 2.5
LIMIAR_DESLOCAMENTO_VERTICAL = 30.0
LIMIAR_VELOCIDADE_VERTICAL = 45.0
TEMPO_MAXIMO_TRANSICAO = 5.0
MODO_TESTE_SEGURO = True

# No modo experimental seguro, o evento pode ser reproduzido por meio
# de agachamento rápido e inclinação do tronco, sem impacto com o solo.
TEMPO_CONFIRMACAO_DEITADO = 1.0
JANELA_ANALISE_QUEDA = 1.5
LIMIAR_TEMPO_QUEDA_TESTE = 3.5
LIMIAR_DESLOCAMENTO_TESTE = 12.0
LIMIAR_VELOCIDADE_TESTE = 8.0
ASPECTO_MINIMO_TESTE = 0.82
TEMPO_RECUPERACAO_RAPIDA = 3.0
INTERVALO_MINIMO_EVENTOS = 2.0
DURACAO_MENSAGEM = 4.0


# =====================================================================
# Operações geométricas sobre caixas delimitadoras
# =====================================================================

def centro_bbox(bbox):
    x, y, w, h = bbox
    return x + w / 2.0, y + h / 2.0


def area_bbox(bbox):
    return max(0, bbox[2]) * max(0, bbox[3])


def distancia_bboxes(a, b):
    if a is None or b is None:
        return float("inf")

    ax, ay = centro_bbox(a)
    bx, by = centro_bbox(b)
    return ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5


def calcular_iou(a, b):
    if a is None or b is None:
        return 0.0

    ax, ay, aw, ah = a
    bx, by, bw, bh = b

    x1 = max(ax, bx)
    y1 = max(ay, by)
    x2 = min(ax + aw, bx + bw)
    y2 = min(ay + ah, by + bh)

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    uniao = area_bbox(a) + area_bbox(b) - inter

    return inter / uniao if uniao > 0 else 0.0


def bbox_valido(bbox, largura, altura):
    x, y, w, h = bbox

    if w <= 0 or h <= 0:
        return False

    altura_frac = h / float(altura)
    largura_frac = w / float(largura)
    area_frac = (w * h) / float(largura * altura)

    return (
        altura_frac >= ALTURA_MINIMA_PESSOA_FRAC
        and largura_frac >= LARGURA_MINIMA_PESSOA_FRAC
        and area_frac <= AREA_MAXIMA_PESSOA_FRAC
        and x >= 0
        and y >= 0
        and x + w <= largura
        and y + h <= altura
    )


# =====================================================================
# Detecção neural e associação temporal de pessoas
# =====================================================================

def carregar_detector():
    dispositivo = torch.device("cpu")

    pesos = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
    modelo = ssdlite320_mobilenet_v3_large(weights=pesos)
    modelo.to(dispositivo)
    modelo.eval()

    preprocessamento = pesos.transforms()

    return modelo, preprocessamento, dispositivo


@torch.inference_mode()
def detectar_pessoas(
    frame_bgr,
    modelo,
    preprocessamento,
    dispositivo,
):
    """
    Executa a inferência no quadro de entrada e retorna apenas as detecções
    correspondentes à classe COCO ``person`` (índice 1), filtradas pelos
    limiares de confiança e validade geométrica.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    tensor = torch.from_numpy(frame_rgb).permute(2, 0, 1)
    tensor = preprocessamento(tensor).to(dispositivo)

    resultado = modelo([tensor])[0]

    caixas = resultado["boxes"].cpu().numpy()
    classes = resultado["labels"].cpu().numpy()
    confiancas = resultado["scores"].cpu().numpy()

    altura, largura = frame_bgr.shape[:2]
    pessoas = []

    for caixa, classe, confianca in zip(
        caixas,
        classes,
        confiancas,
    ):
        if int(classe) != 1:
            continue

        if float(confianca) < CONFIANCA_MINIMA_PESSOA:
            continue

        x1, y1, x2, y2 = caixa
        bbox = (
            max(0, int(x1)),
            max(0, int(y1)),
            min(largura, int(x2)) - max(0, int(x1)),
            min(altura, int(y2)) - max(0, int(y1)),
        )

        if not bbox_valido(bbox, largura, altura):
            continue

        pessoas.append(
            {
                "bbox": bbox,
                "confianca": float(confianca),
            }
        )

    return pessoas


def escolher_pessoa(pessoas, bbox_anterior):
    if not pessoas:
        return None

    # Na aquisição inicial, seleciona a detecção com maior confiança e área.
    if bbox_anterior is None:
        return max(
            pessoas,
            key=lambda p: (
                p["confianca"],
                area_bbox(p["bbox"]),
            ),
        )

    candidatos = []

    for pessoa in pessoas:
        bbox = pessoa["bbox"]
        sobreposicao = calcular_iou(bbox, bbox_anterior)
        distancia = distancia_bboxes(bbox, bbox_anterior)

        if (
            sobreposicao < IOU_MINIMO_RASTREAMENTO
            and distancia > DISTANCIA_MAXIMA_RASTREAMENTO
        ):
            continue

        pontuacao = (
            3.0 * sobreposicao
            + pessoa["confianca"]
            + 1.0 / (1.0 + distancia / 100.0)
        )
        candidatos.append((pontuacao, pessoa))

    if not candidatos:
        return None

    return max(candidatos, key=lambda item: item[0])[1]


# =====================================================================
# Estimativa geométrica de postura e classificação de eventos
# =====================================================================

def classificar_postura(bbox, altura_em_pe_referencia):
    """
    Estima a postura a partir das dimensões da caixa delimitadora de uma
    pessoa previamente confirmada pelo detector neural.
    """
    _, _, w, h = bbox
    aspecto = w / float(max(h, 1))

    altura_relativa = None
    if altura_em_pe_referencia is not None:
        altura_relativa = h / max(altura_em_pe_referencia, 1.0)

    # Identifica uma configuração predominantemente horizontal.
    if aspecto >= ASPECTO_DEITADO_MIN:
        return "deitado", aspecto, altura_relativa

    # Verifica redução significativa da altura em relação à referência em pé.
    if (
        altura_relativa is not None
        and altura_relativa <= ALTURA_DEITADO_MAX_FRAC_EM_PE
        and aspecto >= 0.95
    ):
        return "deitado", aspecto, altura_relativa

    # Identifica uma configuração corporal mais baixa e compacta.
    if (
        aspecto > ASPECTO_EM_PE_MAX
        or (
            altura_relativa is not None
            and altura_relativa <= ALTURA_AGACHADO_MAX_FRAC_EM_PE
        )
    ):
        return "agachado", aspecto, altura_relativa

    return "em_pe", aspecto, altura_relativa


def registrar_evento(
    caminho,
    evento,
    caminho_posturas,
    duracao,
    deslocamento,
    velocidade,
    confianca,
):
    with open(caminho, "a", newline="", encoding="utf-8") as arquivo:
        csv.writer(arquivo).writerow(
            [
                dt.datetime.now().isoformat(timespec="seconds"),
                evento,
                caminho_posturas,
                f"{duracao:.2f}",
                f"{deslocamento:.1f}",
                f"{velocidade:.1f}",
                f"{confianca:.2f}",
            ]
        )



def avaliar_queda(
    duracao,
    deslocamento,
    velocidade,
    aspecto,
    estado_final,
):
    """
    Avalia os parâmetros temporais, cinemáticos e geométricos do evento.

    No modo experimental seguro, a decisão utiliza uma regra de pontuação.
    No modo normal, todos os critérios definidos devem ser satisfeitos.
    O estado postural posterior ao início da análise não participa da decisão.
    """
    if MODO_TESTE_SEGURO:
        criterio_tempo = duracao <= LIMIAR_TEMPO_QUEDA_TESTE
        criterio_deslocamento = (
            deslocamento >= LIMIAR_DESLOCAMENTO_TESTE
        )
        criterio_velocidade = (
            velocidade >= LIMIAR_VELOCIDADE_TESTE
        )
        criterio_postura = aspecto >= ASPECTO_MINIMO_TESTE

        pontos = sum(
            [
                criterio_tempo,
                criterio_deslocamento,
                criterio_velocidade,
                criterio_postura,
            ]
        )

        # No modo experimental seguro, três dos quatro critérios são suficientes.
        queda = pontos >= 3
    else:
        criterio_tempo = duracao <= LIMIAR_TEMPO_QUEDA
        criterio_deslocamento = (
            deslocamento >= LIMIAR_DESLOCAMENTO_VERTICAL
        )
        criterio_velocidade = (
            velocidade >= LIMIAR_VELOCIDADE_VERTICAL
        )
        criterio_postura = aspecto >= ASPECTO_DEITADO_MIN
        pontos = sum(
            [
                criterio_tempo,
                criterio_deslocamento,
                criterio_velocidade,
                criterio_postura,
            ]
        )
        queda = all(
            [
                criterio_tempo,
                criterio_deslocamento,
                criterio_velocidade,
                criterio_postura,
            ]
        )

    criterios = {
        "tempo": criterio_tempo,
        "deslocamento": criterio_deslocamento,
        "velocidade": criterio_velocidade,
        "postura": criterio_postura,
        "pontos": pontos,
    }

    return queda, criterios


def imprimir_analise_terminal(
    duracao,
    deslocamento,
    velocidade,
    aspecto,
    estado_final,
    criterios,
):
    """Apresenta no terminal o relatório técnico da análise realizada."""
    resultado_criterio = lambda valor: (
        "ATENDIDO" if valor else "NÃO ATENDIDO"
    )

    print("\n" + "=" * 72)
    print("RELATÓRIO TÉCNICO DA ANÁLISE DO EVENTO")
    print("=" * 72)
    print("Estado postural posterior........: desconsiderado na classificação")
    print(f"Duração da janela de análise.....: {duracao:.2f} s")
    print(f"Deslocamento vertical máximo.....: {deslocamento:.1f} px")
    print(f"Velocidade vertical máxima.......: {velocidade:.1f} px/s")
    print(f"Razão máxima largura/altura.......: {aspecto:.2f}")
    print("-" * 72)
    print("AVALIAÇÃO DOS CRITÉRIOS")
    print(
        "Critério temporal.................: "
        f"{resultado_criterio(criterios['tempo'])}"
    )
    print(
        "Critério de deslocamento..........: "
        f"{resultado_criterio(criterios['deslocamento'])}"
    )
    print(
        "Critério de velocidade............: "
        f"{resultado_criterio(criterios['velocidade'])}"
    )
    print(
        "Critério geométrico...............: "
        f"{resultado_criterio(criterios['postura'])}"
    )
    print(
        f"Critérios satisfeitos..............: {criterios['pontos']}/4"
    )
    print("=" * 72)


# =====================================================================
# Fluxo principal de aquisição, análise e registro
# =====================================================================

def main():
    print("=" * 72)
    print("SISTEMA EXPERIMENTAL DE DETECÇÃO DE QUEDAS")
    print("=" * 72)
    print("Inicializando o modelo neural de detecção de pessoas...")
    modelo, preprocessamento, dispositivo = carregar_detector()
    print("Modelo.............................: SSDLite320 MobileNetV3")
    print("Dispositivo de processamento......: CPU")
    print("Estado do detector.................: inicializado com sucesso")

    if MODO_TESTE_SEGURO:
        print("Modo de avaliação..................: experimental seguro")
        print(
            "Procedimento recomendado...........: agachamento rápido com "
            "inclinação controlada do tronco"
        )
        print(
            "Observação metodológica............: a postura posterior ao "
            "evento não participa da classificação"
        )

    print("=" * 72)
    print("Sistema disponível para aquisição e análise de vídeo.\n")

    captura = cv2.VideoCapture(INDICE_CAMERA)
    if not captura.isOpened():
        raise RuntimeError("Não foi possível acessar a webcam.")

    largura = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))

    pasta_downloads = Path.home() / "Downloads"
    pasta_downloads.mkdir(parents=True, exist_ok=True)

    caminho_video = pasta_downloads / "video_detector_pessoas.mp4"
    caminho_log = pasta_downloads / "log_detector_pessoas.csv"

    gravador = cv2.VideoWriter(
        str(caminho_video),
        cv2.VideoWriter_fourcc(*"mp4v"),
        FPS_SAIDA,
        (largura, altura),
    )

    if not gravador.isOpened():
        captura.release()
        raise RuntimeError("Não foi possível criar o vídeo de saída.")

    with caminho_log.open("w", newline="", encoding="utf-8") as arquivo:
        csv.writer(arquivo).writerow(
            [
                "timestamp",
                "evento",
                "caminho_posturas",
                "duracao_s",
                "deslocamento_vertical_px",
                "velocidade_vertical_px_s",
                "confianca_pessoa",
            ]
        )

    numero_frame = 0
    bbox_pessoa = None
    confianca_pessoa = 0.0
    tempo_ultima_pessoa = 0.0

    historico_bbox = deque(maxlen=5)
    historico_altura_em_pe = deque(maxlen=30)
    historico_cy = deque(maxlen=5)

    estado_atual = "indefinido"
    estado_pendente = "indefinido"
    contagem_pendente = 0

    ultimo_tempo_em_pe = None
    ultimo_cy_em_pe = None

    transicao_ativa = False
    tempo_inicio_transicao = None
    cy_inicio_transicao = None
    caminho_transicao = []

    gatilho_ativo = False
    tempo_gatilho = None
    estado_que_disparou = None
    queda_suspeita = False
    tempo_inicio_deitado = None
    decisao_realizada = False
    max_deslocamento_gatilho = 0.0
    max_velocidade_gatilho = 0.0
    max_aspecto_gatilho = 0.0

    mensagem = ""
    cor_mensagem = (0, 0, 255)
    tempo_mensagem = 0.0
    tempo_ultimo_evento = 0.0

    try:
        while True:
            sucesso, frame = captura.read()
            if not sucesso:
                break

            frame = corrigir_distorcao(frame)

            agora = time.time()
            numero_frame += 1

            # Executa a inferência em intervalos regulares para reduzir o custo computacional.
            if (
                numero_frame % INTERVALO_DETECCAO == 0
                or bbox_pessoa is None
            ):
                pessoas = detectar_pessoas(
                    frame,
                    modelo,
                    preprocessamento,
                    dispositivo,
                )
                escolhida = escolher_pessoa(pessoas, bbox_pessoa)

                if escolhida is not None:
                    bbox_pessoa = escolhida["bbox"]
                    confianca_pessoa = escolhida["confianca"]
                    tempo_ultima_pessoa = agora
                    historico_bbox.append(bbox_pessoa)

                    # Aplica suavização temporal para reduzir oscilações da caixa delimitadora.
                    valores = np.array(historico_bbox, dtype=np.float32)
                    bbox_pessoa = tuple(
                        int(v) for v in np.median(valores, axis=0)
                    )

            pessoa_presente = (
                bbox_pessoa is not None
                and agora - tempo_ultima_pessoa
                <= TEMPO_MAXIMO_SEM_PESSOA
            )

            if not pessoa_presente:
                bbox_pessoa = None
                confianca_pessoa = 0.0
                historico_bbox.clear()
                historico_cy.clear()

                estado_instantaneo = "indefinido"
                aspecto = 0.0
                altura_relativa = None
            else:
                altura_em_pe_referencia = (
                    float(np.median(historico_altura_em_pe))
                    if historico_altura_em_pe
                    else None
                )

                (
                    estado_instantaneo,
                    aspecto,
                    altura_relativa,
                ) = classificar_postura(
                    bbox_pessoa,
                    altura_em_pe_referencia,
                )

                _, cy_bruto = centro_bbox(bbox_pessoa)
                historico_cy.append(cy_bruto)
                cy_atual = float(np.median(historico_cy))

                # Atualiza a referência antropométrica somente quando a postura
                # em pé apresenta estabilidade suficiente.
                if estado_instantaneo == "em_pe":
                    historico_altura_em_pe.append(bbox_pessoa[3])

            # Aplica histerese temporal para evitar mudanças espúrias de estado.
            if estado_instantaneo == estado_pendente:
                contagem_pendente += 1
            else:
                estado_pendente = estado_instantaneo
                contagem_pendente = 1

            mudou_estado = (
                contagem_pendente >= CONFIRMA_QUADROS
                and estado_pendente != estado_atual
            )

            if mudou_estado:
                estado_anterior = estado_atual
                estado_atual = estado_pendente

                print("\n[TRANSIÇÃO POSTURAL CONFIRMADA]")
                print(
                    f"Estado anterior....................: "
                    f"{estado_anterior.upper()}"
                )
                print(
                    f"Estado atual.......................: "
                    f"{estado_atual.upper()}"
                )
                print(
                    f"Confiança da detecção..............: "
                    f"{confianca_pessoa:.2f}"
                )

                # A transição postural apenas inicia a janela de análise.
                # A classificação subsequente utiliza exclusivamente os parâmetros medidos.
                if (
                    not gatilho_ativo
                    and estado_anterior == "em_pe"
                    and estado_atual in {"agachado", "deitado"}
                ):
                    gatilho_ativo = True
                    transicao_ativa = True
                    tempo_gatilho = agora
                    tempo_inicio_transicao = agora
                    estado_que_disparou = estado_atual
                    caminho_transicao = ["em_pe", estado_atual]
                    decisao_realizada = False

                    if ultimo_cy_em_pe is not None:
                        cy_inicio_transicao = ultimo_cy_em_pe
                    elif pessoa_presente:
                        _, cy_inicio_transicao = centro_bbox(bbox_pessoa)
                    else:
                        cy_inicio_transicao = None

                    max_deslocamento_gatilho = 0.0
                    max_velocidade_gatilho = 0.0
                    max_aspecto_gatilho = aspecto

                    print("\n" + "=" * 72)
                    print("INÍCIO DA ANÁLISE DE EVENTO CANDIDATO")
                    print("=" * 72)
                    print(
                        f"Transição de referência............: "
                        f"EM_PÉ -> {estado_atual.upper()}"
                    )
                    print(
                        f"Janela temporal de aquisição.......: "
                        f"{JANELA_ANALISE_QUEDA:.1f} s"
                    )
                    print(
                        "Parâmetros monitorados..............: tempo, deslocamento, "
                        "velocidade e razão geométrica"
                    )
                    print(
                        "Critério metodológico...............: a postura posterior "
                        "é desconsiderada"
                    )
                    print("=" * 72)

                elif gatilho_ativo:
                    if (
                        not caminho_transicao
                        or caminho_transicao[-1] != estado_atual
                    ):
                        caminho_transicao.append(estado_atual)

            # Durante a janela temporal, conserva os valores máximos observados,
            # independentemente da postura assumida posteriormente.
            if gatilho_ativo and tempo_gatilho is not None:
                duracao_gatilho = agora - tempo_gatilho

                if pessoa_presente and cy_inicio_transicao is not None:
                    _, cy_medicao = centro_bbox(bbox_pessoa)
                    deslocamento_atual = max(
                        0.0,
                        cy_medicao - cy_inicio_transicao,
                    )
                    velocidade_atual = (
                        deslocamento_atual
                        / max(duracao_gatilho, 1e-3)
                    )

                    max_deslocamento_gatilho = max(
                        max_deslocamento_gatilho,
                        deslocamento_atual,
                    )
                    max_velocidade_gatilho = max(
                        max_velocidade_gatilho,
                        velocidade_atual,
                    )
                    max_aspecto_gatilho = max(
                        max_aspecto_gatilho,
                        aspecto,
                    )

                # A classificação é executada somente após o término da janela de análise.
                if duracao_gatilho >= JANELA_ANALISE_QUEDA:
                    queda_detectada, criterios = avaliar_queda(
                        duracao_gatilho,
                        max_deslocamento_gatilho,
                        max_velocidade_gatilho,
                        max_aspecto_gatilho,
                        estado_atual,
                    )

                    imprimir_analise_terminal(
                        duracao_gatilho,
                        max_deslocamento_gatilho,
                        max_velocidade_gatilho,
                        max_aspecto_gatilho,
                        "IGNORADO",
                        criterios,
                    )

                    print("\n" + "=" * 72)
                    print("CLASSIFICAÇÃO FINAL DO EVENTO")
                    print("=" * 72)

                    if queda_detectada:
                        evento = "EVENTO_COMPATIVEL_COM_QUEDA"
                        print(
                            "Resultado...........................: "
                            "EVENTO COMPATÍVEL COM QUEDA"
                        )
                        print(
                            "Interpretação......................: os parâmetros "
                            "satisfizeram a regra de classificação"
                        )
                    else:
                        evento = "EVENTO_NAO_CLASSIFICADO_COMO_QUEDA"
                        print(
                            "Resultado...........................: "
                            "EVENTO NÃO CLASSIFICADO COMO QUEDA"
                        )
                        print(
                            "Interpretação......................: os parâmetros "
                            "não satisfizeram a regra de classificação"
                        )

                    print("=" * 72)

                    caminho_texto = " -> ".join(caminho_transicao)

                    if (
                        agora - tempo_ultimo_evento
                        >= INTERVALO_MINIMO_EVENTOS
                    ):
                        registrar_evento(
                            caminho_log,
                            evento,
                            caminho_texto,
                            duracao_gatilho,
                            max_deslocamento_gatilho,
                            max_velocidade_gatilho,
                            confianca_pessoa,
                        )
                        tempo_ultimo_evento = agora

                    print("\n[REGISTRO DO EVENTO]")
                    print(f"Identificador da classe............: {evento}")
                    print(
                        f"Duração observada...................: "
                        f"{duracao_gatilho:.2f} s"
                    )
                    print(
                        f"Deslocamento vertical máximo.......: "
                        f"{max_deslocamento_gatilho:.1f} px"
                    )
                    print(
                        f"Velocidade vertical máxima.........: "
                        f"{max_velocidade_gatilho:.1f} px/s"
                    )
                    print(
                        f"Razão máxima largura/altura.........: "
                        f"{max_aspecto_gatilho:.2f}"
                    )
                    print("Situação do registro...............: concluído")
                    print("=" * 72 + "\n")

                    decisao_realizada = True
                    gatilho_ativo = False
                    transicao_ativa = False
                    tempo_gatilho = None
                    estado_que_disparou = None
                    queda_suspeita = False
                    tempo_inicio_deitado = None
                    caminho_transicao = []
                    max_deslocamento_gatilho = 0.0
                    max_velocidade_gatilho = 0.0
                    max_aspecto_gatilho = 0.0

            # Atualiza a referência em pé somente fora de uma análise em andamento.
            if (
                pessoa_presente
                and estado_atual == "em_pe"
                and not gatilho_ativo
            ):
                _, cy = centro_bbox(bbox_pessoa)
                ultimo_tempo_em_pe = agora
                ultimo_cy_em_pe = cy


            # Renderização da interface de monitoramento.
            if pessoa_presente:
                x, y, w, h = bbox_pessoa

                cor = {
                    "em_pe": (0, 200, 0),
                    "agachado": (255, 200, 0),
                    "deitado": (0, 165, 255),
                    "indefinido": (150, 150, 150),
                }[estado_instantaneo]

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    cor,
                    3,
                )

                cv2.putText(
                    frame,
                    (
                        f"PESSOA {confianca_pessoa:.2f} | "
                        f"{estado_instantaneo}"
                    ),
                    (x, max(25, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    cor,
                    2,
                )

                rel_texto = (
                    f"{altura_relativa:.2f}"
                    if altura_relativa is not None
                    else "-"
                )
                cv2.putText(
                    frame,
                    f"aspecto={aspecto:.2f} altura_rel={rel_texto}",
                    (10, altura - 45),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.50,
                    (255, 255, 255),
                    1,
                )
            else:
                cv2.putText(
                    frame,
                    "Nenhuma pessoa confirmada",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.75,
                    (0, 0, 255),
                    2,
                )

            cv2.putText(
                frame,
                f"Estado confirmado: {estado_atual}",
                (10, altura - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.60,
                (255, 255, 0),
                2,
            )

            status_gatilho = (
                f"Gatilho ativo: {estado_que_disparou}"
                if gatilho_ativo
                else "Gatilho inativo"
            )
            cv2.putText(
                frame,
                status_gatilho,
                (10, altura - 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 220, 255) if gatilho_ativo else (180, 180, 180),
                2,
            )

            gravador.write(frame)
            cv2.imshow(
                "Detector neural de pessoas e quedas - q para sair",
                frame,
            )

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        captura.release()
        gravador.release()
        cv2.destroyAllWindows()

    print("\n" + "=" * 72)
    print("ENCERRAMENTO DO SISTEMA")
    print("=" * 72)
    print(f"Arquivo de vídeo...................: {caminho_video}")
    print(f"Arquivo de registro................: {caminho_log}")
    print("Estado final........................: execução encerrada corretamente")
    print("=" * 72)


if __name__ == "__main__":
    main()

### Declaração do Uso de Inteligência Artificial Generativa (IAG)

Em estrito cumprimento à **Portaria CNPq nº 2.664/2026** (Artigo 2º, alíneas *c*, *d* e *f*), que regulamenta a integridade da pesquisa científica e a transparência no uso de ferramentas tecnológicas emergentes, declaramos o emprego de Inteligência Artificial Generativa (IAG) no desenvolvimento deste projeto.

---

#### 1. Mapeamento de Ferramentas e Finalidades

| Ferramenta de IAG | Fase do Desenvolvimento | Finalidade Específica |
| :--- | :--- | :--- |
| **Claude** (*Anthropic*) | Desenvolvimento, Análise e Redação | Apoio na estruturação, refatoração, sintaxe e otimização de algoritmos em Python / Jupyter Notebook, além do auxílio na formulação, síntese de dados e revisão técnica do relatório final. |

---

#### 2. Formatos dos Artefatos do Projeto

A documentação e os entregáveis deste trabalho estão distribuídos nos seguintes formatos estipulados:

* **Relatório Técnico e Documentação:** Arquivos em formato Jupyter Notebook (`.ipynb`) e exportação final formatada em documento `.pdf`.
* **Código-Fonte e Algoritmos:** Scripts executáveis em Python (`.py`) e fluxos interativos em Jupyter Notebook (`.ipynb`).
* **Registros de Vídeo e Demonstração:** Registros de inferência e testes com voluntários em formato de vídeo `.mp4`.
* **Apresentação de Resultados:** Conjunto de slides estruturado em formato `.pptx`.

---

#### 3. Termo de Responsabilidade de Autoria e Integridade

* **Autoria e Controle Humano:** Todo o escopo conceitual, a modelagem dos algoritmos de visão computacional, os critérios cinemáticos para detecção de quedas e a condução dos testes práticos foram idealizados, executados e verificados de forma autônoma pelos autores deste projeto.
* **Originalidade do Conteúdo:** Nenhuma saída gerada de forma automatizada por IAG foi incorporada ao trabalho sem a prévia validação, edição crítica e supervisão dos alunos responsáveis.
* **Responsabilidade Integral:** Os autores assumem total responsabilidade pelo conteúdo do relatório e dos códigos apresentados, garantindo a conformidade acadêmica, a ausência de plágio e a precisão técnica das análises expostas, em alinhamento com as metas de aprendizado de Visão Computacional.